# 1. Train a YOLO26 fish body/head detector

This notebook trains the detector that runs on the ESP32-S3, step by step. It is the same recipe
as `scripts/train_yolo26.py` (use that for unattended or cluster runs, e.g. with `hpc/train.job`).

**What it produces:** `runs/<name>/weights/best.pt` (the checkpoint with the best val mAP),
training curves, and val/test scores. The next notebook, `02_quantise_for_esp32.ipynb`, turns
that checkpoint into an int8 model for the microcontroller.

**Needs:** `pip install -r requirements.txt`, the dataset in `dataset/` (or `RABBITFISH_DATASET`),
and a CUDA GPU for real training (the nano model at 320 px trained in about 30 min on an H100,
353 epochs before early stopping). On a laptop, use quick mode to check that everything works.

In [ ]:
import os, sys
from pathlib import Path

# The notebooks live in notebooks/; the shared code is the rabbitfish package at the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from rabbitfish.paths import DATASET, SAMPLES, RUNS, MODELS, FIRMWARE, WEIGHTS  # also sets YOLO_CONFIG_DIR

# Quick mode (RABBITFISH_QUICK=1) shrinks every expensive step so the whole notebook runs in a few
# minutes. It is how the notebooks are tested; the numbers it produces are meaningless.
QUICK = os.environ.get("RABBITFISH_QUICK") == "1"
print("repository:", ROOT)
print("dataset:   ", DATASET, "(found)" if (DATASET / "data.yaml").exists() else "(MISSING: see data/README.md)")
print("quick mode:", QUICK)

## Step 1: look at the data

YOLO format: `dataset/<split>/images/*.jpg` with one `dataset/<split>/labels/*.txt` per image,
each line `class cx cy w h` (normalised to 0-1). Two classes: `0 fishbody`, `1 fishhead`.
`train` is used for training, `val` to pick the best epoch, and `test` (a separate survey day) is
kept for the final, unbiased score.

In [ ]:
import numpy as np
import yaml
from rabbitfish.data import images, load_labels, label_path

data_yaml = DATASET / "data.yaml"
cfg = yaml.safe_load(data_yaml.read_text())
print("classes:", cfg["names"])

for split in ["train", "val", "test"]:
    paths = images(split)
    counts = np.zeros(len(cfg["names"]), int)
    for p in paths:
        lbl = label_path(p)
        if lbl.exists() and lbl.stat().st_size:
            counts += np.bincount(np.loadtxt(lbl, ndmin=2)[:, 0].astype(int), minlength=len(counts))
    print(f"{split:5s}: {len(paths):4d} images, objects per class {dict(zip(cfg['names'].values(), counts.tolist()))}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

COLORS = {0: (232, 89, 12), 1: (25, 113, 194)}  # fishbody orange, fishhead blue (RGB), as on the device's page
NAMES = {0: "fishbody", 1: "fishhead"}

def show(img_bgr, boxes=(), classes=(), scores=None, title="", ax=None):
    """Draw xyxy boxes (in img_bgr's pixels) on an image."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    lw = max(2, img.shape[1] // 400)
    for i, (b, c) in enumerate(zip(boxes, classes)):
        x1, y1, x2, y2 = map(int, b)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[int(c)], lw)
        label = NAMES[int(c)] + (f" {scores[i]:.2f}" if scores is not None else "")
        cv2.putText(img, label, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, lw / 3, COLORS[int(c)], lw)
    ax = ax or plt.figure(figsize=(8, 6)).gca()
    ax.imshow(img); ax.set_title(title); ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, p in zip(axes, images("train")[::300][:3]):
    img = cv2.imread(str(p))
    cls, boxes = load_labels(p, img.shape)
    show(img, boxes, cls, title=f"{p.name}: {len(cls)} labelled objects", ax=ax)
plt.tight_layout()

## Step 2: choose the model and the training settings

- **Model size `n` (nano, 2.5 M parameters)**: on this data the `s` and `m` models are not more
  accurate (`results/model_comparison.md`), and only the nano model fits an ESP32-S3 at a usable speed.
- **Image size 320**: the device runs the model at 320x256, so training at 320 makes it
  see fish at the size it will meet them on the device. 320 px is as accurate as 640 px here.
- The rest is the recipe the YOLO11 baselines used, so results are comparable: AdamW, lr 0.01,
  cosine schedule, batch 8, up to 500 epochs with early stopping after 100 epochs without improvement,
  multi-scale training (0.5x-1.5x), seed 0, COCO-pretrained weights.

In [ ]:
import torch

SIZE, IMGSZ, BATCH = "n", 320, 8
EPOCHS, PATIENCE = (1, 1) if QUICK else (500, 100)
DEVICE = "0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
NAME = f"yolo26{SIZE}-b{BATCH}-i{IMGSZ}" + ("-quick" if QUICK else "")
print(f"run {NAME} on {DEVICE}: {EPOCHS} epochs max")

## Step 3: train

Ultralytics downloads the COCO-pretrained `yolo26n.pt` into `weights/` the first time (on a
cluster without internet on the compute nodes, run that download once on the login node).
Training writes everything to `runs/<NAME>/`; it can be interrupted and resumed from
`runs/<NAME>/weights/last.pt` with `YOLO(last).train(resume=True)`.

In [ ]:
from ultralytics import YOLO, settings

WEIGHTS.mkdir(exist_ok=True)
settings.update({"weights_dir": str(WEIGHTS)})
pretrained = WEIGHTS / f"yolo26{SIZE}.pt"
model = YOLO(str(pretrained) if pretrained.exists() else pretrained.name)

model.train(
    data=str(data_yaml), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    optimizer="AdamW", lr0=0.01, cos_lr=True, multi_scale=0.5, seed=0,
    workers=2 if QUICK else 8, device=DEVICE, project=str(RUNS), name=NAME, exist_ok=True,
    fraction=0.05 if QUICK else 1.0, plots=True,
)
run_dir = RUNS / NAME
best = run_dir / "weights" / "best.pt"
print("best checkpoint:", best)

## Step 4: training curves

The box loss falls on both splits and val mAP50-95 levels off; the checkpoint with the best val
fitness is `best.pt`. (`results/training/` has these curves for every model in the comparison.)

In [ ]:
import pandas as pd

hist = pd.read_csv(run_dir / "results.csv")
hist.columns = [c.strip() for c in hist.columns]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, cols, title in [
    (axes[0], ["train/box_loss", "val/box_loss"], "box loss"),
    (axes[1], ["train/cls_loss", "val/cls_loss"], "class loss"),
    (axes[2], ["metrics/mAP50(B)", "metrics/mAP50-95(B)"], "val mAP"),
]:
    for c in cols:
        if c in hist:
            ax.plot(hist["epoch"], hist[c], label=c)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()

## Step 5: score the checkpoint on val and on the held-out test split

`ultralytics val` at the training image size (`rect=True` keeps the 4:3 frames unpadded). The
deployed model scored mAP50-95 0.687 on val and 0.706 on test (`results/model_comparison.md`).

In [ ]:
best_model = YOLO(str(best))
scores = {}
for split in ["val", "test"]:
    m = best_model.val(data=str(data_yaml), split=split, imgsz=IMGSZ, batch=1, rect=True,
                       device=DEVICE, plots=False, verbose=False)
    scores[split] = {"mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3),
                     "fishbody mAP50-95": round(float(m.box.maps[0]), 3),
                     "fishhead mAP50-95": round(float(m.box.maps[1]), 3)}
pd.DataFrame(scores).T

## Step 6: look at predictions

The bundled samples (`data/samples`) are from the test split, so the model never saw them.

In [ ]:
sample_paths = images("samples", root=SAMPLES.parent)[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, p in zip(axes.ravel(), sample_paths):
    r = best_model.predict(str(p), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]
    show(cv2.imread(str(p)), r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(),
         r.boxes.conf.cpu().numpy(), title=p.name, ax=ax)
plt.tight_layout()

**Next:** `02_quantise_for_esp32.ipynb` with `WEIGHTS_PATH = RUNS / NAME / "weights" / "best.pt"`
(or keep the default, the committed `models/yolo26n-i320/best.pt`).